# Laboratorio de preparación para el Examen Parcial

Este laboratorio integra cálculo exacto y numérico de las Unidades 1 y 2. Úsalo después de resolver la guía por escrito. Los objetivos son:

- comprobar Gram-Schmidt y proyecciones sobre subespacios;
- verificar la proyección sobre un hiperplano;
- resolver e interpretar un problema de mínimos cuadrados;
- explorar falta de unicidad de coeficientes;
- comprobar Parseval en la DFT y la energía descartada en la DCT.

## 0. Preparación

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=6, suppress=True)

## 1. Gram-Schmidt exacto

La función siguiente conserva aritmética exacta. Antes de normalizar, verifica que los vectores obtenidos sean ortogonales.

In [ ]:
def gram_schmidt_exacto(vectores):
    ortogonales = []
    for v in vectores:
        w = sp.Matrix(v)
        for q in ortogonales:
            w -= (w.dot(q) / q.dot(q)) * q
        w = sp.simplify(w)
        if w.dot(w) == 0:
            raise ValueError("La familia es linealmente dependiente")
        ortogonales.append(w)
    ortonormales = [sp.simplify(w / sp.sqrt(w.dot(w))) for w in ortogonales]
    return ortogonales, ortonormales

In [ ]:
V = [sp.Matrix([1, 1, 0]), sp.Matrix([1, 0, 1]), sp.Matrix([0, 1, 1])]
W, U = gram_schmidt_exacto(V)
Q = sp.Matrix.hstack(*U)

print("Vectores ortogonales:")
for w in W:
    sp.pprint(w.T)
print("Q^T Q =")
sp.pprint(sp.simplify(Q.T * Q))

assert sp.simplify(Q.T * Q) == sp.eye(3)
assert sp.Matrix.hstack(*V).rank() == Q.rank() == 3

## 2. Del núcleo a la matriz de proyección

Para proyectar sobre $W=\ker(A)$ seguimos cuatro pasos: hallar una base del núcleo, ortonormalizarla, formar $U$ y calcular $P=UU^T$.

In [ ]:
A = sp.Matrix([[1, 1, 0, 1], [0, 1, 1, 1]])
x = sp.Matrix([2, -1, 3, 1])
base_nucleo = A.nullspace()
_, base_on = gram_schmidt_exacto(base_nucleo)
Uker = sp.Matrix.hstack(*base_on)
Pker = sp.simplify(Uker * Uker.T)
p = sp.simplify(Pker * x)
r = sp.simplify(x - p)

print("Base del núcleo:")
for v in base_nucleo:
    sp.pprint(v.T)
print("Matriz de proyección:")
sp.pprint(Pker)
print("Proyección p =", list(p))
print("Residuo r =", list(r))

In [ ]:
assert sp.simplify(Pker.T - Pker) == sp.zeros(4)
assert sp.simplify(Pker * Pker - Pker) == sp.zeros(4)
assert A * p == sp.zeros(2, 1)
assert sp.simplify(Uker.T * r) == sp.zeros(Uker.cols, 1)
assert sp.simplify(p.dot(r)) == 0
print("Verificaciones satisfechas: simetría, idempotencia y ortogonalidad.")

## 3. Proyección sobre un hiperplano afín

Para $H=\{z:a^Tz=b\}$ usamos

$$P_H(x)=x-\frac{a^Tx-b}{a^Ta}a.$$

In [ ]:
def proyectar_hiperplano(x, a, b):
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    if np.allclose(a, 0):
        raise ValueError("El vector normal no puede ser cero")
    p = x - ((a @ x - b) / (a @ a)) * a
    distancia = abs(a @ x - b) / np.linalg.norm(a)
    return p, distancia

xh = np.array([2, 1, 0, -1], dtype=float)
a = np.array([1, -2, 2, -1], dtype=float)
b = 5.0
ph, dh = proyectar_hiperplano(xh, a, b)
print("Proyección:", ph)
print("Distancia:", dh)
print("a^T p:", a @ ph)

assert np.allclose(a @ ph, b)
assert np.allclose(np.linalg.norm(xh - ph), dh)
assert np.linalg.matrix_rank(np.column_stack([xh - ph, a])) == 1

## 4. Mínimos cuadrados: cálculo e interpretación

Ajustaremos una recta a cuatro datos. Numéricamente usamos `lstsq`; las ecuaciones normales se verifican como propiedad teórica.

In [ ]:
t = np.array([0., 1., 2., 3.])
y = np.array([1., 2., 2., 5.])
X = np.column_stack([np.ones_like(t), t])
beta, _, rango, _ = np.linalg.lstsq(X, y, rcond=None)
y_ajustada = X @ beta
residuo = y - y_ajustada

print("beta =", beta)
print("rango(X) =", rango)
print("valores ajustados =", y_ajustada)
print("residuo =", residuo)
print("X.T @ residuo =", X.T @ residuo)

assert rango == X.shape[1]
assert np.allclose(X.T @ residuo, 0)
assert np.allclose(X.T @ X @ beta, X.T @ y)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(t, y, color="#be2d37", label="datos")
ax.plot(t, y_ajustada, color="#14578c", label="ajuste")
for ti, yi, yhi in zip(t, y, y_ajustada):
    ax.plot([ti, ti], [yhi, yi], color="gray", linewidth=1)
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Ajuste lineal y residuos")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### Identidad de optimalidad

Comparamos el minimizador con otro vector de coeficientes $c$.

In [ ]:
c = np.array([0., 1.])
lado_izquierdo = np.linalg.norm(X @ c - y)**2
lado_derecho = (np.linalg.norm(X @ beta - y)**2
                  + np.linalg.norm(X @ (c - beta))**2)
print("Error para c:", lado_izquierdo)
print("Error mínimo + separación:", lado_derecho)
assert np.allclose(lado_izquierdo, lado_derecho)

## 5. Rango deficiente: coeficientes distintos, mismo ajuste

In [ ]:
Xr = np.array([[1., 0., 0.], [1., 1., 2.], [1., 2., 4.], [1., 3., 6.]])
yr = np.array([1., 2., 2., 5.])
beta0 = np.linalg.lstsq(Xr, yr, rcond=None)[0]
n = np.array([0., -2., 1.])
beta1 = beta0 + 3 * n

print("rango(Xr) =", np.linalg.matrix_rank(Xr))
print("beta0 =", beta0)
print("beta1 =", beta1)
print("diferencia de ajustes =", np.max(np.abs(Xr @ beta0 - Xr @ beta1)))

assert np.allclose(Xr @ n, 0)
assert np.allclose(Xr @ beta0, Xr @ beta1)

## 6. DFT y Parseval

Usamos la normalización $\widehat x_k=M^{-1}\sum_n x_ne^{-2\pi i kn/M}$. Por ello `fft` debe dividirse entre $M$.

In [ ]:
senal = np.array([1., 0., -1., 0.])
M = len(senal)
n_idx = np.arange(M)
k_idx = np.arange(M)[:, None]
F = np.exp(-2j * np.pi * k_idx * n_idx / M) / M
coef = F @ senal
reconstruida = np.array([np.sum(coef * np.exp(2j * np.pi * np.arange(M) * n / M))
                          for n in range(M)])
energia_muestras = np.mean(np.abs(senal)**2)
energia_frecuencias = np.sum(np.abs(coef)**2)

print("Coeficientes DFT:", np.round(coef, 12))
print("Reconstrucción:", np.real_if_close(reconstruida))
print("Energías:", energia_muestras, energia_frecuencias)

assert np.allclose(coef, np.fft.fft(senal) / M)
assert np.allclose(reconstruida, senal)
assert np.allclose(energia_muestras, energia_frecuencias)

## 7. DCT: energía conservada y error

Construimos una DCT-II ortonormal sin depender de SciPy.

In [ ]:
def matriz_dct(N):
    k = np.arange(N)[:, None]
    n = np.arange(N)[None, :]
    C = np.cos(np.pi * (2 * n + 1) * k / (2 * N))
    C[0, :] *= np.sqrt(1 / N)
    C[1:, :] *= np.sqrt(2 / N)
    return C

N = 16
C = matriz_dct(N)
ii, jj = np.indices((N, N))
imagen = 40 + 3 * ii + 2 * jj + 35 * ((ii // 4 + jj // 4) % 2)
B = C @ imagen @ C.T
K = 5
BK = np.zeros_like(B)
BK[:K, :K] = B[:K, :K]
aprox = C.T @ BK @ C
error = np.linalg.norm(imagen - aprox, ord="fro")**2
energia_descartada = np.sum((B - BK)**2)

print("Error cuadrático:", error)
print("Energía descartada:", energia_descartada)
print("Fracción de coeficientes conservados:", K**2 / N**2)

assert np.allclose(C @ C.T, np.eye(N))
assert np.allclose(C.T @ B @ C, imagen)
assert np.allclose(np.linalg.norm(imagen, "fro"), np.linalg.norm(B, "fro"))
assert np.allclose(error, energia_descartada)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(imagen, cmap="gray")
axes[0].set_title("Imagen original")
axes[1].imshow(np.log1p(np.abs(B)), cmap="magma")
axes[1].set_title("log(1 + |DCT|)")
axes[2].imshow(aprox, cmap="gray")
axes[2].set_title(f"Reconstrucción {K}×{K}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Actividades de cierre

1. Cambia el orden de los vectores del primer ejemplo. ¿Cambia la base ortonormal? ¿Cambia el subespacio?
2. En la proyección sobre el núcleo, comprueba que $x-p$ pertenece al espacio fila de $A$.
3. Sustituye el hiperplano por uno que pase por el origen y compara las fórmulas.
4. Agrega una columna duplicada a la matriz del ajuste lineal y describe todas las soluciones.
5. Conserva en la DCT los 25 coeficientes de mayor magnitud. Compara su error con el bloque $5\times5$ y explica el resultado.
6. Redacta, sin código, la identidad teórica que verifica cada `assert` del cuaderno.